# 03 — Housing Affordability vs. Homeless Rate

Tests whether higher housing costs correlate with higher homelessness rates across states.
- **HUD Fair Market Rent (1BR)**: median market rent by state (2023)
- **Rent burden**: % of renter households paying ≥30% of income on rent (Census ACS 2022)

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_state_data.csv')
df = df.dropna(subset=['homeless_rate_per_10k', 'median_1br_fmr', 'rent_burden_pct'])
print(f'Loaded {len(df)} states for housing affordability analysis')

Loaded 51 states for housing affordability analysis


In [2]:
slope, intercept, r, p, se = stats.linregress(df['median_1br_fmr'], df['homeless_rate_per_10k'])
r2 = r ** 2
print(f'FMR vs homeless rate: r={r:.3f}, R²={r2:.3f}, p={p:.4f}')

x_range = [df['median_1br_fmr'].min(), df['median_1br_fmr'].max()]
y_range = [slope * x + intercept for x in x_range]

fig1 = px.scatter(
    df, x='median_1br_fmr', y='homeless_rate_per_10k', text='state_postal',
    color='homeless_rate_per_10k', color_continuous_scale='Reds',
    title=f'Median 1BR Fair Market Rent vs. Homeless Rate by State<br><sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f}</sup>',
    labels={'median_1br_fmr': 'Median 1BR FMR ($)', 'homeless_rate_per_10k': 'Homeless Rate per 10k'},
    hover_name='state_name',
)
fig1.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression', line=dict(color='darkred', dash='dash')))
fig1.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig1.show()

FMR vs homeless rate: r=0.746, R²=0.557, p=0.0000


In [3]:
slope2, intercept2, r2v, p2, se2 = stats.linregress(df['rent_burden_pct'], df['homeless_rate_per_10k'])
r2_2 = r2v ** 2
print(f'Rent burden vs homeless rate: r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f}')

x_range2 = [df['rent_burden_pct'].min(), df['rent_burden_pct'].max()]
y_range2 = [slope2 * x + intercept2 for x in x_range2]

fig2 = px.scatter(
    df, x='rent_burden_pct', y='homeless_rate_per_10k', text='state_postal',
    color='homeless_rate_per_10k', color_continuous_scale='Reds',
    title=f'Rent Burden (% paying ≥30% income on rent) vs. Homeless Rate<br><sup>r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f}</sup>',
    labels={'rent_burden_pct': 'Rent Burden (%)', 'homeless_rate_per_10k': 'Homeless Rate per 10k'},
    hover_name='state_name',
)
fig2.add_trace(go.Scatter(x=x_range2, y=y_range2, mode='lines', name='Regression', line=dict(color='darkred', dash='dash')))
fig2.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig2.show()

Rent burden vs homeless rate: r=0.456, R²=0.208, p=0.0008


In [4]:
print('\nCorrelation Summary:')
print(f'  FMR vs Homeless Rate: r={r:.3f}, p={p:.4f}')
print(f'  Rent Burden vs Homeless Rate: r={r2v:.3f}, p={p2:.4f}')


Correlation Summary:
  FMR vs Homeless Rate: r=0.746, p=0.0000
  Rent Burden vs Homeless Rate: r=0.456, p=0.0008
